# Legal Document Graph Construction

This notebook builds argument graphs from legal documents with:
- **Sequential edges**: Follows relationships
- **Semantic edges**: LegalBERT-based similarity (support/oppose)
- **Citation edges**: References to legal citations

In [ ]:
!pip install transformers torch networkx scikit-learn tqdm

In [ ]:
import json
from pathlib import Path
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import pickle
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import torch
from tqdm import tqdm
from typing import List, Dict, Optional

### CONFIGURATION

In [ ]:
class GraphConfig:
    # Similarity thresholds
    SIM_THRESHOLD = 0.75
    SIM_WINDOW = 10
    SEMANTIC_TOP_K = 3
    SEMANTIC_MIN_SCORE = 0.75

    # Label edges
    LABEL_WINDOW = 5

    # Processing
    BATCH_SIZE = 32
    MAX_LENGTH = 512

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Paths
    INPUT_DIR = Path("/kaggle/input/annotated-extended")
    OUT_DIR = Path("/kaggle/working/graphs")

    @classmethod
    def setup_directories(cls):
        cls.OUT_DIR.mkdir(exist_ok=True)

config = GraphConfig()
config.setup_directories()

# LOAD BERT MODEL
print(f"🔧 Using device: {config.DEVICE}")
print("📥 Loading InCaseLawBert model...")
tokenizer = AutoTokenizer.from_pretrained("law-ai/InCaseLawBERT")
model = AutoModel.from_pretrained("law-ai/InCaseLawBERT").to(config.DEVICE)
model.eval()
print("✅ Model loaded successfully!")

### EMBEDDING FUNCTIONS

In [ ]:
def get_embeddings_batch(texts: List[str], batch_size: int = 32) -> np.ndarray:
    """
    Process embeddings in batches for efficiency.
    Returns numpy array of shape [num_texts, embedding_dim]
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=config.MAX_LENGTH, padding=True
        ).to(config.DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)
            token_emb = outputs.last_hidden_state  # [batch_size, L, D]
            mask = inputs['attention_mask'].unsqueeze(-1).expand(token_emb.size()).float()
            mean_emb = (token_emb * mask).sum(dim=1) / mask.sum(dim=1)
            all_embeddings.append(mean_emb.cpu().numpy())

    return np.vstack(all_embeddings)

### GRAPH BUILDING FUNCTIONS

In [ ]:
def build_graph_from_doc(doc: Dict) -> Optional[nx.MultiDiGraph]:
    """
    Build base graph from document with nodes and sequential edges.
    Returns None if document has no labeled clauses.
    """
    try:
        clauses = [c for c in doc.get("clauses", [])
                   if c.get("label", "None") != "None"]

        if not clauses:
            print(f"⚠️ Warning: {doc.get('doc_id', 'unknown')} has no labeled clauses")
            return None

        N = len(clauses)
        G = nx.MultiDiGraph()

        # Add nodes
        for i, c in enumerate(clauses):
            cid = c.get("clause_id", i)
            G.add_node(cid, text=c["text"], label=c["label"], idx=i)

        # Sequential edges (follows)
        for i in range(N-1):
            a, b = clauses[i]["clause_id"], clauses[i+1]["clause_id"]
            G.add_edge(a, b, relation="follows", score=1.0)

        return G

    except KeyError as e:
        print(f"❌ Error processing {doc.get('doc_id', 'unknown')}: Missing key {e}")
        return None
    except Exception as e:
        print(f"❌ Error processing {doc.get('doc_id', 'unknown')}: {e}")
        return None

def add_semantic_edges_legalbert(
    G: nx.MultiDiGraph,
    threshold: float = 0.7,
    window: int = 10,
    batch_size: int = 32
) -> nx.MultiDiGraph:
    """
    Add semantic similarity edges using InCaseLawBert embeddings.
    """
    nodes = sorted(G.nodes(data=True), key=lambda x: x[1]['idx'])
    texts = [d['text'] for _, d in nodes]
    ids = [n for n, _ in nodes]
    labels = [d['label'] for _, d in nodes]

    if not texts:
        return G

    print(f"⚙️ Computing LegalBERT embeddings for {len(texts)} clauses (batch size: {batch_size})...")

    embeddings = get_embeddings_batch(texts, batch_size=batch_size)
    sim_matrix = cosine_similarity(embeddings)

    added = 0
    N = len(ids)

    for i in range(N):
        # Determine window bounds
        start_j = max(0, i - window) if window else 0
        end_j = min(N, i + window + 1) if window else N

        for j in range(start_j, end_j):
            if i == j:
                continue

            sim = sim_matrix[i, j]
            if sim >= threshold:
                G.add_edge(
                    ids[i], ids[j],
                    relation="semantic_legalbert",
                    score=float(sim),
                    src_label=labels[i],
                    tgt_label=labels[j]
                )
                added += 1

    print(f"✅ Added {added} semantic edges")
    return G


def filter_semantic_edges(G: nx.MultiDiGraph, k: int = 3, min_score: float = 0.75) -> nx.MultiDiGraph:
    """
    Keep only top-k semantic edges per node in BOTH directions (outgoing and incoming).
    Uses (neighbor, key) to identify edges.
    Each node ends up with at most k outgoing and at most k incoming semantic edges.
    """
    def is_semantic(d):
        return "semantic" in d.get("relation", "")

    to_remove = []

    # Cap Outgoing
    for node in G.nodes():
        sem_edges = []
        for _, v, key, d in G.out_edges(node, data=True, keys=True):
            if is_semantic(d):
                sem_edges.append((v, key, d.get("score", 0.0)))
        sem_edges.sort(key=lambda x: x[2], reverse=True)
        keep_out = {(v, key) for v, key, score in sem_edges[:k] if score >= min_score}
        for _, v, key, d in G.out_edges(node, data=True, keys=True):
            if is_semantic(d) and (v, key) not in keep_out:
                to_remove.append((node, v, key))

    for u, v, key in to_remove:
        if G.has_edge(u, v, key):
            G.remove_edge(u, v, key)
    to_remove.clear()

    # Cap Incoming
    for node in G.nodes():
        sem_edges = []
        for u, _, key, d in G.in_edges(node, data=True, keys=True):
            if is_semantic(d):
                sem_edges.append((u, key, d.get("score", 0.0)))
        sem_edges.sort(key=lambda x: x[2], reverse=True)
        keep_in = {(u, key) for u, key, score in sem_edges[:k] if score >= min_score}
        for u, _, key, d in G.in_edges(node, data=True, keys=True):
            if is_semantic(d) and (u, key) not in keep_in:
                to_remove.append((u, node, key))

    for u, v, key in to_remove:
        if G.has_edge(u, v, key):
            G.remove_edge(u, v, key)

    return G


def transform_semantic_edges(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    Transform semantic edges to green_support/red_opposes based on node labels.
    """
    for u, v, key, d in list(G.edges(data=True, keys=True)):
        if "semantic" not in d.get("relation", ""):
            continue

        src_label = G.nodes[u]['label']
        tgt_label = G.nodes[v]['label']

        # Determine edge type based on labels
        if src_label == "Opposition" or tgt_label == "Opposition":
            edge_type = "red_opposes"
        else:
            edge_type = "green_support"

        # Update edge attributes
        d['relation'] = edge_type
        d['color'] = "green" if "green" in edge_type else "red"
        d['style'] = "dashed"
        d['source'] = 'semantic'

    print("✅ Semantic edges transformed into logic-based support/opposes edges")
    return G


def merge_graph(G_master: nx.MultiDiGraph, G_new: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    Merge edges from G_new into G_master without adding duplicates
    of the same relation type between the same (u,v).
    """
    for u, v, key, data_new in G_new.edges(data=True, keys=True):
        relation_new = data_new.get("relation")

        # Get all existing edges between u and v
        edge_dict = G_master.get_edge_data(u, v, default={})
        duplicate = False
        for _, data_existing in edge_dict.items():
            if data_existing.get("relation") == relation_new:
                duplicate = True
                break

        # Add edge if no duplicate relation exists
        if not duplicate:
            G_master.add_edge(u, v, **data_new)

    return G_master


def print_graph_stats(G: nx.MultiDiGraph, doc_id: str):
    """Print detailed graph statistics."""
    print(f"\n📊 Graph Statistics: {doc_id}")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")

    # Edge counts by relation
    edge_counts = {}
    for _, _, d in G.edges(data=True):
        rel = d.get('relation', 'unknown')
        edge_counts[rel] = edge_counts.get(rel, 0) + 1

    print("  Edges by relation:")
    for rel, count in sorted(edge_counts.items()):
        print(f"    {rel}: {count}")

    # Node label distribution
    label_counts = {}
    for _, d in G.nodes(data=True):
        label = d.get('label', 'None')
        label_counts[label] = label_counts.get(label, 0) + 1

    print("  Node labels:")
    for label, count in sorted(label_counts.items()):
        print(f"    {label}: {count}")

### PROCESS DOCUMENTS - BUILD BASE GRAPHS

In [ ]:
def process_document(doc: Dict, add_semantic: bool = False) -> Optional[nx.MultiDiGraph]:
    """
    Process a single document into a graph.
    """
    # Build base graph
    G = build_graph_from_doc(doc)
    if G is None:
        return None

    # Add semantic edges
    if add_semantic:
        G = add_semantic_edges_legalbert(
            G, threshold=config.SIM_THRESHOLD,
            window=config.SIM_WINDOW, batch_size=config.BATCH_SIZE
        )
        G = filter_semantic_edges(
            G, k=config.SEMANTIC_TOP_K,
            min_score=config.SEMANTIC_MIN_SCORE
        )
        G = transform_semantic_edges(G)

    return G


PICKLE_OUT = config.OUT_DIR / "pickles"
PICKLE_OUT.mkdir(exist_ok=True, parents=True)

print("\n" + "=" * 60)
print("PROCESSING DOCUMENTS - BASE GRAPHS")
print("=" * 60)

json_files = list(config.INPUT_DIR.rglob("*.json"))
if not json_files:
    print(f"⚠️ No JSON files found in {config.INPUT_DIR}")
else:
    for p in tqdm(json_files, desc="Processing documents"):
        try:
            with open(p, "r", encoding="utf-8") as f:
                doc = json.load(f)

            G = process_document(doc, add_semantic=False)

            if G is not None:
                # Save graph
                with open(PICKLE_OUT / f"{doc['doc_id']}.gpickle", "wb") as f:
                    pickle.dump(G, f)

                print(f"✅ {doc['doc_id']}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        except Exception as e:
            print(f"❌ Error processing {p}: {e}")

print("\n" + "=" * 60)
print("BASE GRAPH CONSTRUCTION COMPLETE")
print("=" * 60)

### EXPORT BASE GRAPHS TO JSON

In [ ]:
GRAPH_JSON_DIR = config.OUT_DIR / "graph_jsons"
GRAPH_JSON_DIR.mkdir(exist_ok=True, parents=True)

print("\n" + "=" * 60)
print("EXPORTING BASE GRAPHS TO JSON")
print("=" * 60)

for gpickle_file in tqdm(list(PICKLE_OUT.glob("*.gpickle")), desc="Exporting graphs"):
    try:
        with open(gpickle_file, "rb") as f:
            G = pickle.load(f)

        # Nodes with their attributes
        nodes_data = []
        for n, d in G.nodes(data=True):
            nodes_data.append({
                "id": n,
                "text": d.get("text", ""),
                "label": d.get("label", "None"),
                "idx": d.get("idx", None)
            })

        # Edges with attributes
        edges_data = []
        for u, v, key, d in G.edges(data=True, keys=True):
            edges_data.append({
                "source": u,
                "target": v,
                "relation": d.get("relation"),
                "score": d.get("score", None)
            })

        # Full graph JSON
        graph_json = {
            "doc_id": gpickle_file.stem,
            "nodes": nodes_data,
            "edges": edges_data
        }

        # Save as JSON
        out_file = GRAPH_JSON_DIR / f"{gpickle_file.stem}_graph.json"
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(graph_json, f, indent=2, ensure_ascii=False)

        print(f"✅ {gpickle_file.stem}: {len(nodes_data)} nodes, {len(edges_data)} edges")
    except Exception as e:
        print(f"❌ Error exporting {gpickle_file}: {e}")

print("\n" + "=" * 60)
print("BASE GRAPH EXPORT COMPLETE")
print("=" * 60)

### ADD SEMANTIC EDGES TO GRAPHS

In [ ]:
SEMANTIC_DIR = config.OUT_DIR / "semantic_graphs"
SEMANTIC_DIR.mkdir(exist_ok=True, parents=True)

print("\n" + "=" * 60)
print("ADDING SEMANTIC EDGES TO GRAPHS\n")
print(f"Threshold: {config.SIM_THRESHOLD}, Window: {config.SIM_WINDOW}")
print(f"Top-K: {config.SEMANTIC_TOP_K}, Min Score: {config.SEMANTIC_MIN_SCORE}")
print("=" * 60)


gpickle_files = list(PICKLE_OUT.glob("*.gpickle"))
if not gpickle_files:
    print("⚠️ No graph files found to process")
else:
    for gpickle_file in tqdm(gpickle_files, desc="Adding semantic edges"):
        try:
            # Load original base graph
            with open(gpickle_file, "rb") as f:
                G_orig = pickle.load(f)

            # Add semantic edges
            G_sem = add_semantic_edges_legalbert(
                G_orig.copy(),
                threshold=config.SIM_THRESHOLD,
                window=config.SIM_WINDOW,
                batch_size=config.BATCH_SIZE
            )

            # Filter top-k semantic edges
            G_sem = filter_semantic_edges(
                G_sem,
                k=config.SEMANTIC_TOP_K,
                min_score=config.SEMANTIC_MIN_SCORE
            )

            # Transform semantic edges to green_support/red_opposes
            G_sem = transform_semantic_edges(G_sem)

            # Merge base edges back into semantic graph
            G_final = merge_graph(G_sem, G_orig)

            # Save combined graph
            out_pickle = SEMANTIC_DIR / gpickle_file.name
            with open(out_pickle, "wb") as f:
                pickle.dump(G_final, f)

            # Export to JSON for visualization
            nodes_data = [
                {"id": n, "text": d.get("text", ""), "label": d.get("label")}
                for n, d in G_final.nodes(data=True)
            ]

            edges_data = [
                {
                    "source": u,
                    "target": v,
                    "relation": d.get("relation"),
                    "score": d.get("score", None)
                }
                for u, v, key, d in G_final.edges(data=True, keys=True)
            ]

            graph_json = {
                "doc_id": gpickle_file.stem,
                "nodes": nodes_data,
                "edges": edges_data
            }

            out_json = SEMANTIC_DIR / f"{gpickle_file.stem}_semantic.json"
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(graph_json, f, indent=2, ensure_ascii=False)

            # Print statistics
            print_graph_stats(G_final, gpickle_file.stem)

        except Exception as e:
            print(f"❌ Error processing {gpickle_file}: {e}")

print("\n" + "=" * 60)
print("SEMANTIC EDGE ADDITION COMPLETE")
print("=" * 80)

### VISUALIZATION OF GRAPH

In [ ]:
def visualize_graph(graph_json_path: str, output_path: str = "graph_visualization.png"):
    """
    Visualize an intra- or inter-document clause-level argument graph.
    Supports external (cited) nodes and inter-document citation edges.
    """

    # Load graph JSON
    with open(graph_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    G = nx.MultiDiGraph()

    # Add nodes
    for node in data["nodes"]:
        G.add_node(
            node["id"],
            label=node.get("label", ""),
            text=node.get("text", ""),
            external=node.get("external", False),
            cited_doc=node.get("cited_doc", None)
        )

    # Add edges with styles
    for edge in data["edges"]:
        rel = edge.get("relation", "")

        # Defaults
        color = "black"
        style = "solid"
        width = 2

        if rel == "label_support":
            color, style = "#2ca02c", "solid"

        elif rel == "green_support":
            color, style = "#98df8a", "dashed"

        elif rel == "red_opposes":
            color, style = "#ff9896", "dashed"

        elif rel == "follows":
            color, style = "gray", "dotted"
            width = 1.5

        elif rel == "inter_citation":
            color, style = "#7b3294", "solid"
            width = 3  # emphasize inter-doc edges

        G.add_edge(
            edge["source"],
            edge["target"],
            relation=rel,
            score=edge.get("score"),
            color=color,
            style=style,
            width=width
        )

    # Layout
    pos = nx.spring_layout(G, k=0.9, seed=42)

    # Node colors
    node_colors = []
    for _, d in G.nodes(data=True):
        if d.get("external", False):
            node_colors.append("#ffd966")  # cited clause
        else:
            lbl = d.get("label", "")
            if lbl == "Claim":
                node_colors.append("lightgreen")
            elif lbl == "Premise":
                node_colors.append("lightblue")
            elif lbl == "Opposition":
                node_colors.append("lightcoral")
            else:
                node_colors.append("lightgray")

    # Node labels
    labels = {}
    for n, d in G.nodes(data=True):
        if d.get("external", False):
            labels[n] = f"{n}\n[CITED]\n{d.get('cited_doc', '')}"
        else:
            labels[n] = f"{n}\n{d.get('label', '')}"


    # Plot graph
    plt.figure(figsize=(30, 30))

    nx.draw_networkx_nodes(
        G, pos,
        node_color=node_colors,
        node_size=700,
        edgecolors="black"
    )

    nx.draw_networkx_labels(
        G, pos,
        labels,
        font_size=9
    )

    # Draw edges individually
    for u, v, d in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            edge_color=d["color"],
            style=d["style"],
            width=d.get("width", 2),
            arrowsize=18
        )

    # Legend
    legend_elements = [
        mlines.Line2D([], [], color='lightgreen', marker='o', linestyle='None',
                      label='Claim (current doc)', markersize=10),
        mlines.Line2D([], [], color='lightblue', marker='o', linestyle='None',
                      label='Premise (current doc)', markersize=10),
        mlines.Line2D([], [], color='lightcoral', marker='o', linestyle='None',
                      label='Opposition (current doc)', markersize=10),
        mlines.Line2D([], [], color='#ffd966', marker='o', linestyle='None',
                      label='Cited clause (external)', markersize=10),

        mlines.Line2D([], [], color='#2ca02c', linestyle='solid', label='label_support'),
        mlines.Line2D([], [], color='#98df8a', linestyle='dashed', label='semantic_support'),
        mlines.Line2D([], [], color='#ff9896', linestyle='dashed', label='semantic_opposes'),
        mlines.Line2D([], [], color='gray', linestyle='dotted', label='follows'),
        mlines.Line2D([], [], color='#7b3294', linestyle='solid', label='inter_document_citation'),
    ]

    plt.legend(handles=legend_elements, loc='upper left', fontsize=11)

    # Final touches
    plt.title("Clause-Level Argument Graph (Intra + Inter Document)", fontsize=20)
    plt.axis("off")
    plt.tight_layout()

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"✅ Graph visualization saved as '{output_path}'")

GRAPH_JSON_FILE = str(SEMANTIC_DIR / "SC_07_2012.clauses_semantic.json")
visualize_graph(GRAPH_JSON_FILE, "clause_argument_graph_all_edges.png")

### ADD INTER-DOCUMENT CITATION EDGES

In [ ]:
def normalize_cited_doc_id(cited_doc_id: str, available_graphs: Dict[str, Path]) -> Optional[str]:
    """
    Normalize cited document ID to match graph file naming.
    Examples: SC_CA_50_2008 -> SC_50_2008.clauses
    """
    # Try removing CA prefix
    if cited_doc_id.startswith("SC_CA_"):
        doc_id = cited_doc_id.replace("SC_CA_", "SC_")
        graph_key = f"{doc_id}.clauses"
        if graph_key in available_graphs:
            return graph_key

        fr_doc_id = cited_doc_id.replace("SC_CA_", "SC_FR_")
        graph_key = f"{fr_doc_id}.clauses"
        if graph_key in available_graphs:
            return graph_key

    # Try direct match for SC_FR_ cases
    if cited_doc_id.startswith("SC_FR_"):
        graph_key = f"{cited_doc_id}.clauses"
        if graph_key in available_graphs:
            return graph_key

    graph_key = f"{cited_doc_id}.clauses"
    if graph_key in available_graphs:
        return graph_key

    return None


def add_inter_document_citations():
    """
    Add inter-document citation edges to graphs.
    Only adds the specific cited clauses as external nodes.
    """
    print("\nADDING INTER-DOCUMENT CITATION EDGES\n")

    # Load citation data
    CITATIONS_FILE = Path("/kaggle/input/graph-citations/graph_citations.json")
    if not CITATIONS_FILE.exists():
        print(f"❌ Citations file not found: {CITATIONS_FILE}")
        return

    with open(CITATIONS_FILE, "r", encoding="utf-8") as f:
        citation_data = json.load(f)

    # Get available graph files
    available_graphs = {}
    for graph_file in SEMANTIC_DIR.glob("*.json"):
        doc_id = graph_file.stem.removesuffix(".semantic")
        available_graphs[doc_id] = graph_file

    print(f"📂 Found {len(available_graphs)} graph files")
    print(f"📄 Found {citation_data['metadata']['total_citations_found']} total citations")
    print()

    stats = {
        "graphs_updated": 0,
        "citations_processed": 0,
        "cited_clauses_added": 0,
        "inter_edges_created": 0,
        "citations_to_available_docs": 0
    }

    # Process each document's citations
    for doc_data in tqdm(citation_data["citations_by_document"], desc="Adding inter-document citations"):
        source_doc_id = doc_data["doc_id"]
        citations = doc_data.get("citations", [])

        if not citations:
            continue

        # Load source graph JSON
        source_graph_file = SEMANTIC_DIR / f"{source_doc_id}.semantic.json"
        if not source_graph_file.exists():
            print(f"  ⚠️  Source graph not found: {source_doc_id}")
            continue

        with open(source_graph_file, "r", encoding="utf-8") as f:
            source_graph_data = json.load(f)

        # Convert to NetworkX graph for easier manipulation
        G = nx.MultiDiGraph()

        # Add existing nodes
        for node in source_graph_data["nodes"]:
            G.add_node(
                node["id"],
                text=node.get("text", ""),
                label=node.get("label", "None"),
                external=False
            )

        # Add existing edges
        for edge in source_graph_data["edges"]:
            G.add_edge(
                edge["source"],
                edge["target"],
                relation=edge.get("relation", ""),
                score=edge.get("score", None)
            )

        # Track per-document stats
        doc_edges_created = 0

        # Process citations for this document
        for citation in citations:
            stats["citations_processed"] += 1
            cited_doc_id = citation["cited_doc_id"]
            source_clause_id = citation["source_clause_id"]
            target_clause_id = citation.get("target_clause_id")

            # Only process citations with target_clause_id
            if target_clause_id is None:
                continue

            # Normalize cited document ID
            normalized_doc_id = normalize_cited_doc_id(cited_doc_id, available_graphs)

            if not normalized_doc_id:
                continue

            stats["citations_to_available_docs"] += 1

            # Load target document graph to get the cited clause
            target_graph_file = available_graphs[normalized_doc_id]
            with open(target_graph_file, "r", encoding="utf-8") as f:
                target_graph_data = json.load(f)

            # Find the specific cited clause
            cited_clause = None
            for node in target_graph_data["nodes"]:
                if node.get("id") == target_clause_id:
                    cited_clause = node
                    break

            if not cited_clause:
                print(f"  ⚠️  Cited clause {target_clause_id} not found in {normalized_doc_id}")
                continue

            # Create unique ID for external clause
            external_node_id = f"{normalized_doc_id.replace('.clauses', '')}::clause_{target_clause_id}"

            # Add cited clause as external node
            if not G.has_node(external_node_id):
                G.add_node(
                    external_node_id,
                    text=cited_clause.get("text", ""),
                    label=cited_clause.get("label", "None"),
                    external=True,
                    cited_doc=normalized_doc_id.replace(".clauses", ""),
                    original_clause_id=target_clause_id
                )
                stats["cited_clauses_added"] += 1

            # Update source clause node to indicate it makes a citation
            if G.has_node(source_clause_id):
                # Add citation metadata to the source node
                node_data = G.nodes[source_clause_id]

                # Set cited_doc to indicate which document this clause cites
                node_data["cited_doc"] = normalized_doc_id.replace(".clauses", "")
                node_data["original_clause_id"] = target_clause_id

                # Create inter-document citation edge
                G.add_edge(
                    source_clause_id,
                    external_node_id,
                    relation="inter_citation",
                    citation_text=citation.get("citation_text", ""),
                    similarity_score=citation.get("similarity_score"),
                    color="#7b3294",
                    style="solid",
                    width=3
                )
                doc_edges_created += 1
                stats["inter_edges_created"] += 1

        # Export updated graph back to JSON
        if doc_edges_created > 0:
            nodes_data = []
            for node_id, data in G.nodes(data=True):
                nodes_data.append({
                    "id": node_id,
                    "text": data.get("text", ""),
                    "label": data.get("label", "None"),
                    "external": data.get("external", False),
                    "cited_doc": data.get("cited_doc", None),
                    "original_clause_id": data.get("original_clause_id", None)
                })

            edges_data = []
            for u, v, key, data in G.edges(data=True, keys=True):
                            edges_data.append({
                    "source": u,
                    "target": v,
                    "relation": data.get("relation", ""),
                    "score": data.get("score", None),
                    "citation_text": data.get("citation_text", None),
                    "similarity_score": data.get("similarity_score", None)
                })

            updated_graph_json = {
                "doc_id": source_doc_id,
                "nodes": nodes_data,
                "edges": edges_data
            }

            # Overwrite the semantic graph file
            with open(source_graph_file, "w", encoding="utf-8") as f:
                json.dump(updated_graph_json, f, indent=2, ensure_ascii=False)

            stats["graphs_updated"] += 1
            print(f"  ✅ {source_doc_id}: Added {doc_edges_created} inter-document edges")

    print("\n" + "=" * 80)
    print("INTER-DOCUMENT CITATION STATISTICS")
    print("=" * 80)
    print(f"Graphs updated: {stats['graphs_updated']}")
    print(f"Citations processed: {stats['citations_processed']}")
    print(f"Citations to available documents: {stats['citations_to_available_docs']}")
    print(f"Cited clauses added (external nodes): {stats['cited_clauses_added']}")
    print(f"Inter-document edges created: {stats['inter_edges_created']}")
    print("=" * 80)

add_inter_document_citations()


### VISUALIZE INTER-DOCUMENT CITATION GRAPH

In [ ]:
EXAMPLE_DOC = "SC_07_2012.clauses"
GRAPH_JSON_FILE = SEMANTIC_DIR / f"{EXAMPLE_DOC}.semantic.json"

if GRAPH_JSON_FILE.exists():
    print("=" * 80)
    print(f"VISUALIZING INTER-DOCUMENT CITATION GRAPH: {EXAMPLE_DOC}")
    print("=" * 80)

    # Use the existing visualize_graph function (it already supports external nodes)
    visualize_graph(str(GRAPH_JSON_FILE), f"{EXAMPLE_DOC}_inter_document_graph.png")

    print(f"\n✅ Visualization saved as '{EXAMPLE_DOC}_inter_document_graph.png'")
    print("\nThe graph shows:")
    print("  - Regular nodes (green/blue/coral): Clauses from the current document")
    print("  - Yellow nodes: Cited clauses from other documents (external)")
    print("  - Purple edges: Inter-document citation links")
else:
    print(f"⚠️  Graph file not found: {GRAPH_JSON_FILE}")
    print("   Run the previous cell first to add inter-document citations.")